# E-commerce Recommendation Model Training

This notebook trains a hybrid recommendation model combining:
1. **Collaborative Filtering**: Using Alternating Least Squares (ALS)
2. **Content-Based Filtering**: Using TF-IDF and product features

## Steps:
- Load preprocessed data
- Train ALS model for collaborative filtering
- Compute content-based similarities
- Evaluate model performance
- Save trained model

In [ ]:
# Install required packages
!pip install implicit scikit-learn pandas numpy joblib matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import joblib
from implicit.als import AlternatingLeastSquares
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported successfully!")

## Load Preprocessed Data

In [ ]:
# Load user-item matrix and features
user_item_matrix = np.load('../ml/user_item_matrix.npy')
text_features = np.load('../ml/text_features.npy')

print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"Text features shape: {text_features.shape}")

## Train Collaborative Filtering Model (ALS)

In [ ]:
# Convert to sparse matrix for efficiency
sparse_user_item = csr_matrix(user_item_matrix)

# Train ALS model
model = AlternatingLeastSquares(
    factors=50,
    regularization=0.01,
    iterations=20,
    random_state=42
)

print("Training collaborative filtering model...")
model.fit(sparse_user_item)
print("Training complete!")

## Compute Content-Based Similarities

In [ ]:
# Compute product similarity matrix
print("Computing product similarities...")
product_similarity = cosine_similarity(text_features)
print(f"Similarity matrix shape: {product_similarity.shape}")

# Visualize similarity distribution
plt.figure(figsize=(10, 6))
plt.hist(product_similarity.flatten(), bins=50)
plt.title('Distribution of Product Similarities')
plt.xlabel('Cosine Similarity')
plt.ylabel('Frequency')
plt.show()

## Evaluate Model Performance

In [ ]:
# Example: Get recommendations for user 0
user_id = 0
recommendations = model.recommend(user_id, sparse_user_item[user_id], N=10)

print(f"Top 10 recommendations for user {user_id}:")
for product_id, score in recommendations:
    print(f"  Product {product_id}: Score {score:.4f}")

## Save Trained Model

In [ ]:
# Save the collaborative filtering model
joblib.dump(model, '../ml/model.pkl')
print("Model saved to ml/model.pkl")

# Save product similarity matrix
np.save('../ml/product_similarity.npy', product_similarity)
print("Product similarity saved to ml/product_similarity.npy")

print("\n✅ Model training complete!")
print("Next step: Start the backend API to serve recommendations")